# 2. Tahoe drug-response condition generalization

**Scientific task:** predict the full single-cell count distribution for a completely held-out
$(\mathrm{cell\ line},\mathrm{drug},\mathrm{dose})$ condition from matched DMSO cells.

For condition $c=(\ell,d,a)$,

$$
X_0\sim p_{\rm DMSO}(x\mid \ell,\mathrm{plate}),\qquad
X_1\sim p_{\rm treated}(x\mid \ell,d,a).
$$

The Count Flow Map algorithm is unchanged. This notebook only defines the application,
fair comparison panel, evaluation, and figures.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "countflow").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts import run_scrna_drug_transport as scrna_runner

RUN_SCRIPT = ROOT / "scripts" / "run_scrna_drug_transport.py"
SCRNA_CONFIG = scrna_runner.SCRNA_CONFIG
TAHOE_DATA = scrna_runner.TAHOE_DATA
NFE_VALUES = scrna_runner.NFE_VALUES
UNIT_JUMP_NFE_VALUES = scrna_runner.UNIT_JUMP_NFE_VALUES
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESUME = True
STRICT_PAPER = True

print("device:", DEVICE)
print("data:", TAHOE_DATA)
print("Flow Map / tau-leap grid:", NFE_VALUES)
print("unit-jump grid:", UNIT_JUMP_NFE_VALUES)
print("output:", ROOT / SCRNA_CONFIG["output_dir"])


## Experimental design

All target cells from a validation/test $(\ell,d,a)$ condition are absent from training.
A held-out condition is allowed only when the same cell line, same drug, and exact
cell-line × drug pair remain represented by another dose in training.

All methods therefore receive the **same frozen cells, same 2,000 genes, same split,
and same condition labels**. Test treated cells are used only for final metrics.

Count Flow Map and Count-FM select sampling resolution on validation only. Count Flow Map
and tau-leap use

$$\{1,4,16,64,128,256\},$$

while the original Count-FM unit-jump discretization uses the finer grid

$$\{16,64,128,256,512\}.$$

The main paper table reports Count Flow Map at 1 NFE and at validation-selected NFE.


In [ ]:
print(json.dumps({
    "seeds": SCRNA_CONFIG["seeds"],
    "flowmap_tau_nfe": SCRNA_CONFIG["nfe_values"],
    "unit_jump_nfe": SCRNA_CONFIG["unit_jump_nfe_values"],
    "count_model_training_steps": SCRNA_CONFIG["models"]["count_flow_map"]["steps"],
    "published_baselines": SCRNA_CONFIG["published_baselines"],
    "paper_methods": SCRNA_CONFIG["paper_methods"],
    "supplementary_baselines": SCRNA_CONFIG["builtin_baselines"],
    "selection_metric": SCRNA_CONFIG["selection_metric"],
}, indent=2))


## One-time Tahoe preparation

This experiment uses the versioned `tahoe_condition_holdout.npz`; an older incompatible bundle is intentionally not reused. Preparation selects dose-specific candidate conditions, validates them against native Tahoe Parquet rows, pairs treated cells with same-cell-line plate-matched DMSO controls, performs complete-condition holdout, and selects HVGs from training conditions only.

The selected-Parquet cache is reused whenever its fingerprint is compatible. A new remote expression extraction is needed only when the cached panel is genuinely incompatible.


In [ ]:
# Always call the preparation entry point. It is cheap when the cached bundle is
# current, and it automatically rebuilds stale data after a prep-version change.
cmd = [sys.executable, str(RUN_SCRIPT), "--prepare-only"]
print("Checking/preparing the condition-holdout Tahoe bundle:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

if not TAHOE_DATA.exists():
    raise RuntimeError(f"Preparation did not create {TAHOE_DATA}")


In [ ]:
split_report = pd.read_csv(TAHOE_DATA.parent / "condition_holdout_split_report.csv")
display(split_report.groupby("split").agg(
    n_conditions=("condition_id", "nunique"),
    n_rows=("n_rows", "sum"),
    n_cell_lines=("cell_line_id", "nunique"),
    n_drugs=("drug", "nunique"),
).reset_index())

display(split_report.sort_values(["split", "cell_line_id", "drug", "dose"]).head(30))


## Main comparison models

The formal comparison is self-contained; no legacy third-party environment is required.

- **Count-FM**, with both the original unit-jump sampler and binomial tau-leap, from the
  same trained Count-FM checkpoint.
- **scGen**: public scGen VAE architecture and latent perturbation arithmetic, with the
  log-dose response interpolation used for multi-dose prediction.
- **scVIDR**: the same scGen-family VAE plus its latent dose-response regression principle.
- **CPA**: the released compositional perturbation autoencoder architecture, per-drug
  continuous dosers, covariate/drug embeddings, and adversarial disentanglement.
- **Conditional NB-VAE**: a count-native latent generative baseline with a negative-binomial
  likelihood. It is deliberately not mislabeled as official scVI.
- **Sinkhorn OT + dose interpolation**: a classical distributional transport baseline.
- **Linear dose-response**: a strong classical response baseline.

Nearest-dose / empirical / DMSO controls are retained only as supplementary diagnostics.

The scGen/scVIDR/CPA implementations are based on the public algorithms but are kept inside
this repository so the comparison does not depend on obsolete Python/CUDA environments.
No held-out treated expression is used to fit any baseline.


In [ ]:
paper_methods = pd.DataFrame({"paper_method": SCRNA_CONFIG["paper_methods"]})
display(paper_methods)
print("No external setup step is required.")
print("Existing compatible Count Flow Map / Count-FM checkpoints will be reused because RESUME=True.")


## Train and evaluate

`RESUME=True` means compatible Count Flow Map and Count-FM checkpoints are loaded rather
than retrained. New baseline checkpoints are trained only if they do not already exist or
their data/config fingerprint changed.

Count-FM unit-jump and tau-leap use the **same Count-FM rate network**; only the sampler
changes. All iterative models use validation-only step selection. One-pass baselines appear
as single points in the performance-vs-compute figure and all main methods appear in the
performance-vs-wall-clock figure.

`--strict-paper` verifies that every configured main method actually produced a finite
held-out-test result before the run is marked paper-ready.


In [ ]:
cmd = [sys.executable, str(RUN_SCRIPT), "--device", DEVICE]
if not RESUME:
    cmd.append("--fresh")
if STRICT_PAPER:
    cmd.append("--strict-paper")
print("Running scRNA application:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)
output = ROOT / SCRNA_CONFIG["output_dir"]
print("output:", output)


## Main paper comparison and supplementary diagnostics


In [ ]:
quality = pd.read_csv(output / "main_quality_table_summary.csv")
supplementary = pd.read_csv(output / "supplementary_quality_table_summary.csv")
readiness_path = output / "paper_readiness.json"
readiness = json.loads(readiness_path.read_text()) if readiness_path.exists() else {}
print("paper_ready:", readiness.get("paper_ready", False))
if not readiness.get("paper_ready", False):
    print("missing methods:", readiness.get("missing_paper_methods", []))
    print("non-finite primary metric:", readiness.get("nonfinite_primary_metric_methods", []))

columns = [c for c in [
    "reported_method", "n_runs", "reported_nfe_mean",
    "sliced_w2_mean", "mmd2_rbf_mean",
    "delta_pearson_mean", "delta_spearman_mean",
    "logfc_pearson_mean", "logfc_spearman_mean",
    "deg_logfc_pearson_mean", "deg_logfc_spearman_mean",
    "top_response_gene_overlap_mean", "top_response_direction_accuracy_mean",
    "gene_mean_relative_l1_mean", "gene_variance_relative_l1_mean", "zero_fraction_l1_mean",
    "generation_seconds_mean", "train_seconds_mean",
] if c in quality.columns]

print("Main paper table:")
display(quality[columns].style.format(precision=3, na_rep="—"))
print("Supplementary / diagnostic methods:")
supp_cols = [c for c in columns if c in supplementary.columns]
display(supplementary[supp_cols].style.format(precision=3, na_rep="—"))


## Scientific figures


In [ ]:
for filename in [
    "scrna_method_comparison.png",
    "scrna_representative_conditions.png",
    "scrna_quality_vs_nfe.png",
    "scrna_quality_vs_runtime.png",
]:
    path = output / "figures" / filename
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("missing figure:", path)


# Formal Output

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

summary = pd.read_csv(output / "main_quality_table_summary.csv")
raw = pd.read_csv(output / "main_quality_table_raw.csv")
metrics = pd.read_csv(output / "metrics.csv")

# Replace the validation-selected Count Flow Map row with a genuine fixed-16-NFE row.
cfm16 = metrics[
    (metrics["split"] == "test") &
    (metrics["method"] == "Count Flow Map") &
    (metrics["nfe"] == 16)
].copy()

cfm16_row = {
    "reported_method": "Count Flow Map (16 NFE)",
    "n_runs": len(cfm16),
}

for metric in [
    "sliced_w2",
    "deg_logfc_pearson",
    "top_response_gene_overlap",
    "generation_seconds",
]:
    cfm16_row[f"{metric}_mean"] = cfm16[metric].mean()
    cfm16_row[f"{metric}_std"] = cfm16[metric].std(ddof=1)

summary = summary[
    summary["reported_method"] != "Count Flow Map (validation-selected)"
].copy()

summary = pd.concat(
    [summary, pd.DataFrame([cfm16_row])],
    ignore_index=True,
)


METHOD_ORDER = [
    "Count Flow Map (1 NFE)",
    "Count Flow Map (16 NFE)",
    "Count-FM + unit jump (validation-selected)",
    "Count-FM + binomial tau-leap (validation-selected)",
    "Conditional NB-VAE",
    "CPA",
    "scGen (nearest-dose)",
    "scVIDR",
    "Sinkhorn OT (dose interpolation)",
    "Linear dose-response",
]

DISPLAY_METHOD = {
    "Count Flow Map (1 NFE)": "Count Flow Map (1 NFE)",
    "Count Flow Map (16 NFE)": "Count Flow Map (16 NFE)",
    "Count-FM + unit jump (validation-selected)": "Count-FM + unit jump",
    "Count-FM + binomial tau-leap (validation-selected)": "Count-FM + binomial tau-leap",
    "Conditional NB-VAE": "Conditional NB-VAE",
    "CPA": "CPA",
    "scGen (nearest-dose)": "scGen",
    "scVIDR": "scVIDR",
    "Sinkhorn OT (dose interpolation)": "Sinkhorn OT",
    "Linear dose-response": "Linear dose-response",
}

summary = summary.set_index("reported_method").reindex(METHOD_ORDER).reset_index()

def mean_sd(row, metric, digits=3):
    mean = row.get(f"{metric}_mean", np.nan)
    std = row.get(f"{metric}_std", np.nan)
    if not np.isfinite(mean):
        return "—"
    if np.isfinite(std) and int(row.get("n_runs", 1)) > 1:
        return f"{mean:.{digits}f} ± {std:.{digits}f}"
    return f"{mean:.{digits}f}"

def nfe_text(method):
    if method == "Count Flow Map (16 NFE)":
        return "16"

    if method not in raw["reported_method"].values:
        return "—"
    x = pd.to_numeric(
        raw.loc[raw.reported_method == method, "reported_nfe"],
        errors="coerce",
    ).dropna()

    if len(x) == 0:
        return "1-pass"
    if method in {
        "scGen (nearest-dose)", "scVIDR", "CPA", "Conditional NB-VAE",
        "Sinkhorn OT (dose interpolation)", "Linear dose-response",
    }:
        return "1-pass"

    u = np.sort(x.unique())
    if len(u) == 1:
        return f"{u[0]:g}"
    if "(validation-selected)" in method:
        return f"val.-selected ({x.min():g}–{x.max():g})"
    return f"{x.min():g}–{x.max():g}"    

rows = []
for _, row in summary.iterrows():
    method = row["reported_method"]
    
    rows.append({
        "Method": DISPLAY_METHOD.get(method, method),
        "Inference budget": nfe_text(method),
        "Sliced W2 ↓": mean_sd(row, "sliced_w2"),
        "DEG logFC r ↑": mean_sd(row, "deg_logfc_pearson"),
        "Top-gene overlap ↑": mean_sd(row, "top_response_gene_overlap"),
        "Generation time (s) ↓": mean_sd(row, "generation_seconds", 2),
    })
    
paper_table = pd.DataFrame(rows)

display(
    paper_table.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_properties(subset=["Method"], **{"text-align": "left"})
)

paper_table.to_csv(output / "paper_main_table.csv", index=False)

# Convenient for the manuscript.
print(paper_table.to_latex(index=False, escape=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

summary = pd.read_csv(output / "main_quality_table_summary.csv")
metrics = pd.read_csv(output / "metrics.csv")

# Genuine fixed-16-NFE Count Flow Map result.
cfm16 = metrics[
    (metrics["split"] == "test") &
    (metrics["method"] == "Count Flow Map") &
    (metrics["nfe"] == 16)
].copy()

cfm16_row = {
    "reported_method": "Count Flow Map (16 NFE)",
    "sliced_w2_mean": cfm16["sliced_w2"].mean(),
    "sliced_w2_std": cfm16["sliced_w2"].std(ddof=1),
    "deg_logfc_pearson_mean": cfm16["deg_logfc_pearson"].mean(),
    "deg_logfc_pearson_std": cfm16["deg_logfc_pearson"].std(ddof=1),
    "top_response_gene_overlap_mean": cfm16["top_response_gene_overlap"].mean(),
    "top_response_gene_overlap_std": cfm16["top_response_gene_overlap"].std(ddof=1),
}

summary = summary[
    summary["reported_method"] != "Count Flow Map (validation-selected)"
].copy()

summary = pd.concat(
    [summary, pd.DataFrame([cfm16_row])],
    ignore_index=True,
)

order = [
    "Count Flow Map (1 NFE)",
    "Count Flow Map (16 NFE)",
    "Count-FM + unit jump (validation-selected)",
    "Count-FM + binomial tau-leap (validation-selected)",
    "Conditional NB-VAE",
    "CPA",
    "scGen (nearest-dose)",
    "scVIDR",
    "Sinkhorn OT (dose interpolation)",
    "Linear dose-response",
]

labels = {
    "Count Flow Map (1 NFE)": "Count Flow Map\n(1 NFE)",
    "Count Flow Map (16 NFE)": "Count Flow Map\n(16 NFE)",
    "Count-FM + unit jump (validation-selected)": "Count-FM + unit jump",
    "Count-FM + binomial tau-leap (validation-selected)": "Count-FM + binomial τ-leap",
    "Conditional NB-VAE": "Conditional NB-VAE",
    "CPA": "CPA",
    "scGen (nearest-dose)": "scGen",
    "scVIDR": "scVIDR",
    "Sinkhorn OT (dose interpolation)": "Sinkhorn OT",
    "Linear dose-response": "Linear dose-response",
}

d = summary.set_index("reported_method").reindex(order)
y = np.arange(len(d))

panels = [
    ("sliced_w2", r"Sliced $W_2$ ↓"),
    ("deg_logfc_pearson", "DEG logFC Pearson ↑"),
    ("top_response_gene_overlap", "Top-gene overlap ↑"),
]

fig, axes = plt.subplots(1, 3, figsize=(9.6, 4.8), sharey=True)

for ax, (metric, title) in zip(axes, panels):
    value = d[f"{metric}_mean"].to_numpy(float)
    err = d.get(
        f"{metric}_std",
        pd.Series(np.nan, index=d.index),
    ).fillna(0).to_numpy(float)

    bars = ax.barh(y, value, xerr=err, capsize=2)

    xmax = np.max(value + err)
    label_offset = 0.010 * xmax

    for bar, v, e in zip(bars, value, err):
        ax.text(
            v + e + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.3f}",
            va="center",
            ha="left",
            fontsize=7,
        )    
    

    ax.set_xlim(right=1.22 * xmax)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.2)
    ax.tick_params(labelsize=8)

axes[0].set_yticks(y)
axes[0].set_yticklabels([labels[x] for x in d.index], fontsize=8)
axes[0].invert_yaxis()

fig.tight_layout()
fig.savefig(output / "figures" / "paper_scrna_method_comparison.pdf",
            bbox_inches="tight")
fig.savefig(output / "figures" / "paper_scrna_method_comparison.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

metrics = pd.read_csv(output / "metrics.csv")

methods = [
    "Count Flow Map",
    "Count-FM + unit jump",
    "Count-FM + binomial tau-leap",
]

labels = {
    "Count Flow Map": "Count Flow Map",
    "Count-FM + unit jump": "Count-FM, unit jump",
    "Count-FM + binomial tau-leap": "Count-FM, τ-leap",
}

m = metrics[
    (metrics["split"] == "test") &
    (metrics["method"].isin(methods)) &
    metrics["nfe"].notna()
].copy()

g = (
    m.groupby(["method", "nfe"])
    .agg(
        w2=("sliced_w2", "mean"),
        w2_sd=("sliced_w2", "std"),
        runtime=("generation_seconds", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), sharey=True)

# Quality vs NFE
for method in methods:
    d = g[g.method == method].sort_values("nfe")
    axes[0].errorbar(
        d["nfe"], d["w2"],
        yerr=d["w2_sd"],
        marker="o", linewidth=1.5, capsize=2,
        label=labels[method],
    )

axes[0].set_xscale("log", base=2)
axes[0].set_xlabel("NFE")
axes[0].set_ylabel(r"Sliced $W_2$ ↓")
axes[0].set_title("Quality vs inference budget", fontsize=10)
axes[0].grid(alpha=0.2)
axes[0].legend(fontsize=7)
cfm = g[g["method"] == "Count Flow Map"].set_index("nfe")

for nfe, text in [(1, "1 NFE"), (16, "16 NFE")]:
    row = cfm.loc[float(nfe)]
    axes[0].annotate(
        text,
        xy=(nfe, row["w2"]),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=7,
    )

# Quality vs actual runtime
for method in methods:
    d = g[g.method == method].sort_values("runtime")
    axes[1].plot(
        d["runtime"], d["w2"],
        marker="o", linewidth=1.5,
        label=labels[method],
    )

axes[1].set_xscale("log")
axes[1].set_xlabel("Generation time (s)")
axes[1].set_title("Quality vs generation time", fontsize=10)
axes[1].grid(alpha=0.2)
for nfe, text in [(1, "1 NFE"), (16, "16 NFE")]:
    row = cfm.loc[float(nfe)]
    axes[1].annotate(
        text,
        xy=(row["runtime"], row["w2"]),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=7,
    )

fig.tight_layout()
fig.savefig(output / "figures" / "paper_scrna_efficiency.pdf",
            bbox_inches="tight")
fig.savefig(output / "figures" / "paper_scrna_efficiency.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import torch

from countflow.application_data import load_pair_bundle
from countflow.application_metrics import fit_count_pca
from countflow.application_training import generate_conditional_flow_map
from countflow import application_runner as app_runner

bundle = load_pair_bundle(TAHOE_DATA)

# Training-only PCA, identical evaluation representation.
pca_cfg = SCRNA_CONFIG["pca"]
projection = fit_count_pca(
    torch.cat([bundle.train.x0, bundle.train.x1], dim=0),
    n_components=int(pca_cfg["n_components"]),
    max_cells=int(pca_cfg["max_cells"]),
    seed=int(pca_cfg["seed"]),
)

# Load the already-trained CFM checkpoint. No training occurs here.
seed = int(SCRNA_CONFIG["figure_seed"])
device = torch.device(DEVICE)

flow = app_runner._build_flow_map(
    bundle,
    SCRNA_CONFIG["models"]["count_flow_map"],
).to(device)

checkpoint = output / "checkpoints" / str(seed) / "count_flow_map.pt"
payload = torch.load(checkpoint, map_location=device, weights_only=False)
flow.load_state_dict(payload["state_dict"])
flow.eval()

# Use the fixed 16-NFE operating point shown in the main paper figure.
nfe = 16
assert nfe in NFE_VALUES

# Objective visualization choice only:
# one strongest and one median-strength held-out perturbation.
strength = []
for cid in sorted(bundle.test.condition_id.unique().tolist()):
    mask = bundle.test.condition_id == int(cid)
    ctl = bundle.test.x0[mask].double().mean(0)
    tgt = bundle.test.x1[mask].double().mean(0)
    effect = torch.log1p(tgt) - torch.log1p(ctl)
    strength.append((float(torch.linalg.norm(effect)), int(cid)))

strength.sort()
condition_ids = [
    strength[-1][1],
    strength[len(strength) // 2][1],
]

lookup = {
    int(x["condition_id"]): x
    for x in bundle.metadata["conditions"]
}

rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 2, figsize=(7.6, 5.6))

for row, cid in enumerate(condition_ids):
    mask = bundle.test.condition_id == cid
    control = bundle.test.x0[mask]
    target = bundle.test.x1[mask]

    generated = generate_conditional_flow_map(
        flow,
        control,
        bundle.test.context[mask],
        n_steps=nfe,
        tau=float(SCRNA_CONFIG["tau"]),
        device=DEVICE,
        batch_size=int(SCRNA_CONFIG["generation_batch_size"]),
    )

    # PCA distribution
    ax = axes[row, 0]
    sets = [
        (control, "DMSO"),
        (target, "Observed"),
        (generated, "Count Flow Map, 16 NFE"),
    ]

    for values, label in sets:
        p = projection.transform(values)[:, :2].numpy()
        if len(p) > 250:
            p = p[rng.choice(len(p), 250, replace=False)]
        ax.scatter(p[:, 0], p[:, 1], s=7, alpha=0.40, label=label)

    name = lookup[cid]["condition_name"]
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("PC1", fontsize=8)
    ax.set_ylabel("PC2", fontsize=8)
    ax.tick_params(labelsize=7)

    if row == 0:
        ax.legend(fontsize=7, frameon=False)

    # Perturbation-effect recovery
    ctl_mean = control.double().mean(0)
    observed = (
        torch.log1p(target.double().mean(0)) -
        torch.log1p(ctl_mean)
    ).numpy()
    predicted = (
        torch.log1p(generated.double().mean(0)) -
        torch.log1p(ctl_mean)
    ).numpy()

    r = np.corrcoef(observed, predicted)[0, 1]

    ax = axes[row, 1]
    ax.scatter(observed, predicted, s=5, alpha=0.35)

    lo = min(observed.min(), predicted.min())
    hi = max(observed.max(), predicted.max())
    ax.plot([lo, hi], [lo, hi], "--", linewidth=1)

    ax.set_title(f"Effect recovery, r = {r:.2f}", fontsize=9)
    ax.set_xlabel("Observed effect", fontsize=8)
    ax.set_ylabel("Generated effect", fontsize=8)
    ax.tick_params(labelsize=7)

fig.tight_layout()
fig.savefig(output / "figures" / "paper_scrna_examples.pdf",
            bbox_inches="tight")
fig.savefig(output / "figures" / "paper_scrna_examples.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Compact validation / baseline audit
val_diag = pd.read_csv(output / "validation_diagnostics.csv")
cols = [c for c in ["method", "seed", "sliced_w2", "deg_logfc_pearson", "top_response_gene_overlap"] if c in val_diag.columns]
display(val_diag[cols].sort_values(["method", "seed"]).reset_index(drop=True))

audit_path = output / "baseline_prediction_audit.csv"
if audit_path.exists():
    display(pd.read_csv(audit_path))
